In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/leonidkulyk/ip102-yolov5/IP102_YOLOv5/ip102.yaml
/kaggle/input/datasets/leonidkulyk/ip102-yolov5/IP102_YOLOv5/labels/val/IP027000391.txt
/kaggle/input/datasets/leonidkulyk/ip102-yolov5/IP102_YOLOv5/labels/val/IP095000112.txt
/kaggle/input/datasets/leonidkulyk/ip102-yolov5/IP102_YOLOv5/labels/val/IP029000397.txt
/kaggle/input/datasets/leonidkulyk/ip102-yolov5/IP102_YOLOv5/labels/val/IP043000284.txt
/kaggle/input/datasets/leonidkulyk/ip102-yolov5/IP102_YOLOv5/labels/val/IP013000261.txt
/kaggle/input/datasets/leonidkulyk/ip102-yolov5/IP102_YOLOv5/labels/val/IP101000242.txt
/kaggle/input/datasets/leonidkulyk/ip102-yolov5/IP102_YOLOv5/labels/val/IP063000229.txt
/kaggle/input/datasets/leonidkulyk/ip102-yolov5/IP102_YOLOv5/labels/val/IP014000645.txt
/kaggle/input/datasets/leonidkulyk/ip102-yolov5/IP102_YOLOv5/labels/val/IP052000189.txt
/kaggle/input/datasets/leonidkulyk/ip102-yolov5/IP102_YOLOv5/labels/val/IP008000039.txt
/kaggle/input/datasets/leonidkulyk/ip102-yolov5/

In [10]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
CUDA Available: True
GPU: Tesla T4


In [14]:
!pip install -q ultralytics

In [15]:
from ultralytics import YOLO
import ultralytics

print("Ultralytics Version:", ultralytics.__version__)

Ultralytics Version: 8.4.93


In [16]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root)
    if len(files) > 0:
        print("Sample files:", files[:5])
    print("-" * 60)

/kaggle/input
------------------------------------------------------------
/kaggle/input/datasets
------------------------------------------------------------
/kaggle/input/datasets/leonidkulyk
------------------------------------------------------------
/kaggle/input/datasets/leonidkulyk/ip102-yolov5
------------------------------------------------------------
/kaggle/input/datasets/leonidkulyk/ip102-yolov5/IP102_YOLOv5
Sample files: ['ip102.yaml']
------------------------------------------------------------
/kaggle/input/datasets/leonidkulyk/ip102-yolov5/IP102_YOLOv5/labels
------------------------------------------------------------
/kaggle/input/datasets/leonidkulyk/ip102-yolov5/IP102_YOLOv5/labels/val
Sample files: ['IP027000391.txt', 'IP095000112.txt', 'IP029000397.txt', 'IP043000284.txt', 'IP013000261.txt']
------------------------------------------------------------
/kaggle/input/datasets/leonidkulyk/ip102-yolov5/IP102_YOLOv5/labels/train
Sample files: ['IP071000469.txt', 'IP01

In [20]:
yaml_path = "/kaggle/input/datasets/leonidkulyk/ip102-yolov5/IP102_YOLOv5/ip102.yaml"

with open(yaml_path, "r") as f:
    print(f.read())

train: ./IP102_YOLOv5/images/train/
val: ./IP102_YOLOv5/images/val/

# number of classes
nc: 102

# class names
names: ['rice leaf roller', 'rice leaf caterpillar', 'paddy stem maggot', 'asiatic rice borer', 'yellow rice borer',
        'rice gall midge', 'Rice Stemfly', 'brown plant hopper', 'white backed plant hopper', 'small brown plant hopper',
        'rice water weevil', 'rice leafhopper', 'grain spreader thrips', 'rice shell pest', 'grub', 'mole cricket', 'wireworm',
        'white margined moth', 'black cutworm', 'large cutworm', 'yellow cutworm', 'red spider', 'corn borer', 'army worm', 'aphids',
        'Potosiabre vitarsis', 'peach borer', 'english grain aphid', 'green bug', 'bird cherry-oataphid', 'wheat blossom midge',
        'penthaleus major', 'longlegged spider mite', 'wheat phloeothrips', 'wheat sawfly', 'cerodonta denticornis', 'beet fly',
        'flea beetle', 'cabbage army worm', 'beet army worm', 'Beet spot flies', 'meadow moth', 'beet weevil', 'sericaorient alis

In [21]:
import yaml

kaggle_yaml = {
    "path": "/kaggle/input/datasets/leonidkulyk/ip102-yolov5/IP102_YOLOv5",
    "train": "images/train",
    "val": "images/val",
    "nc": 102,
    "names": [
        'rice leaf roller', 'rice leaf caterpillar', 'paddy stem maggot', 'asiatic rice borer',
        'yellow rice borer', 'rice gall midge', 'Rice Stemfly', 'brown plant hopper',
        'white backed plant hopper', 'small brown plant hopper', 'rice water weevil',
        'rice leafhopper', 'grain spreader thrips', 'rice shell pest', 'grub',
        'mole cricket', 'wireworm', 'white margined moth', 'black cutworm',
        'large cutworm', 'yellow cutworm', 'red spider', 'corn borer',
        'army worm', 'aphids', 'Potosiabre vitarsis', 'peach borer',
        'english grain aphid', 'green bug', 'bird cherry-oataphid',
        'wheat blossom midge', 'penthaleus major', 'longlegged spider mite',
        'wheat phloeothrips', 'wheat sawfly', 'cerodonta denticornis',
        'beet fly', 'flea beetle', 'cabbage army worm', 'beet army worm',
        'Beet spot flies', 'meadow moth', 'beet weevil',
        'sericaorient alismots chulsky', 'alfalfa weevil', 'flax budworm',
        'alfalfa plant bug', 'tarnished plant bug', 'Locustoidea',
        'lytta polita', 'legume blister beetle', 'blister beetle',
        'therioaphis maculata Buckton', 'odontothrips loti', 'Thrips',
        'alfalfa seed chalcid', 'Pieris canidia', 'Apolygus lucorum',
        'Limacodidae', 'Viteus vitifoliae', 'Colomerus vitis',
        'Brevipoalpus lewisi McGregor', 'oides decempunctata',
        'Polyphagotars onemus latus', 'Pseudococcus comstocki Kuwana',
        'parathrene regalis', 'Ampelophaga', 'Lycorma delicatula',
        'Xylotrechus', 'Cicadella viridis', 'Miridae',
        'Trialeurodes vaporariorum', 'Erythroneura apicalis',
        'Papilio xuthus', 'Panonchus citri McGregor',
        'Phyllocoptes oleiverus ashmead', 'Icerya purchasi Maskell',
        'Unaspis yanonensis', 'Ceroplastes rubens',
        'Chrysomphalus aonidum', 'Parlatoria zizyphus Lucus',
        'Nipaecoccus vastalor', 'Aleurocanthus spiniferus',
        'Tetradacus c Bactrocera minax', 'Dacus dorsalis(Hendel)',
        'Bactrocera tsuneonis', 'Prodenia litura', 'Adristyrannus',
        'Phyllocnistis citrella Stainton', 'Toxoptera citricidus',
        'Toxoptera aurantii', 'Aphis citricola Vander Goot',
        'Scirtothrips dorsalis Hood', 'Dasineura sp',
        'Lawana imitata Melichar', 'Salurnis marginella Guerr',
        'Deporaus marginatus Pascoe', 'Chlumetia transversa',
        'Mango flat beak leafhopper', 'Rhytidodera bowrinii white',
        'Sternochetus frigidus', 'Cicadellidae'
    ]
}

with open("ip102_kaggle.yaml", "w") as f:8
    yaml.dump(kaggle_yaml, f, sort_keys=False)

print("Created ip102_kaggle.yaml")

Created ip102_kaggle.yaml


In [22]:
from ultralytics import YOLO

# Load YOLO11 Nano model
model = YOLO("yolo11n.pt")

# Train
results = model.train(
    data="ip102_kaggle.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    workers=2,
    device=0,
    project="AgriGuard_AI",
    name="Training_V2",
    exist_ok=True,
    pretrained=True,
    cache=False,
    verbose=True
)

Ultralytics 8.4.93 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=ip102_kaggle.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=Training_V2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pa

In [23]:
import os

path = "/kaggle/working/runs/detect/AgriGuard_AI/Training_V2"

print("Exists:", os.path.exists(path))
print()

for root, dirs, files in os.walk(path):
    print(root)
    for file in files:
        print("   ", file)

Exists: True

/kaggle/working/runs/detect/AgriGuard_AI/Training_V2
    args.yaml
    train_batch2.jpg
    val_batch0_labels.jpg
    results.csv
    BoxP_curve.png
    val_batch2_labels.jpg
    BoxF1_curve.png
    val_batch0_pred.jpg
    val_batch1_pred.jpg
    confusion_matrix_normalized.png
    val_batch2_pred.jpg
    confusion_matrix.png
    train_batch44121.jpg
    BoxPR_curve.png
    train_batch1.jpg
    results.png
    BoxR_curve.png
    labels.jpg
    val_batch1_labels.jpg
    train_batch44122.jpg
    train_batch44120.jpg
    train_batch0.jpg
/kaggle/working/runs/detect/AgriGuard_AI/Training_V2/weights
    last.pt
    best.pt


In [24]:
!zip -r AgriGuard_AI_Training_V2.zip /kaggle/working/runs/detect/AgriGuard_AI/Training_V2

  adding: kaggle/working/runs/detect/AgriGuard_AI/Training_V2/ (stored 0%)
  adding: kaggle/working/runs/detect/AgriGuard_AI/Training_V2/args.yaml (deflated 52%)
  adding: kaggle/working/runs/detect/AgriGuard_AI/Training_V2/train_batch2.jpg (deflated 5%)
  adding: kaggle/working/runs/detect/AgriGuard_AI/Training_V2/val_batch0_labels.jpg (deflated 14%)
  adding: kaggle/working/runs/detect/AgriGuard_AI/Training_V2/results.csv (deflated 67%)
  adding: kaggle/working/runs/detect/AgriGuard_AI/Training_V2/BoxP_curve.png (deflated 7%)
  adding: kaggle/working/runs/detect/AgriGuard_AI/Training_V2/val_batch2_labels.jpg (deflated 11%)
  adding: kaggle/working/runs/detect/AgriGuard_AI/Training_V2/BoxF1_curve.png (deflated 8%)
  adding: kaggle/working/runs/detect/AgriGuard_AI/Training_V2/weights/ (stored 0%)
  adding: kaggle/working/runs/detect/AgriGuard_AI/Training_V2/weights/last.pt (deflated 10%)
  adding: kaggle/working/runs/detect/AgriGuard_AI/Training_V2/weights/best.pt (deflated 10%)
  addi